# RAG con ChromaDB, agente ReAct, LangSmith y RAGAS (Simón Bolívar)

**Qué pide la tarea y qué hace este notebook**

1. **RAG**: textos sobre Simón Bolívar → embeddings → **Chroma** (BD vectorial) → recuperación por similitud → respuesta del LLM con ese contexto.
2. **Agente ReAct**: el modelo decide cuándo llamar a la herramienta que consulta Chroma (`create_agent` / fallback `create_react_agent`).
3. **LangSmith**: trazas con `LANGCHAIN_API_KEY` + `LANGCHAIN_TRACING_V2` (no es la misma clave que `OPENAI_API_KEY`).
4. **RAGAS**: al menos dos preguntas evaluadas con `faithfulness`, `answer_relevancy`, `context_precision`.
5. **Ejemplo obligatorio**: la pregunta *«¿Quién es Simón Bolívar?»* se resuelve con el pipeline (embedding de la consulta → búsqueda en Chroma → respuesta).

Los embeddings usan **`text-embedding-3-small`** por API de OpenAI (misma `OPENAI_API_KEY` que el chat). Si el enunciado exige *MiniLM local*, usa un venv aparte con `sentence-transformers`.


## 1. Dependencias

Instala paquetes ligeros (sin SciPy ni `sentence-transformers`). Lo ideal es un **venv** y `pip install -r requirements.txt` en la carpeta del proyecto; la celda siguiente también instala con el mismo Python del kernel.

**Reinicia el kernel** después de esta celda si antes tenías errores de imports.


In [1]:
import subprocess
import sys
from pathlib import Path

req = Path.cwd() / "requirements.txt"
cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "chromadb",
    "python-dotenv",
    "langchain",
    "langchain-openai",
    "langchain-chroma",
    "langchain-community",
    "langgraph",
    "datasets",
    "ragas",
    "ipywidgets",
    "mcp",
    "langchain-mcp-adapters",
]
if req.is_file():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
else:
    subprocess.check_call(cmd)

importlib_ok = True
try:
    import chromadb  # noqa: F401
    from langchain_chroma import Chroma  # noqa: F401
    from langchain_openai import OpenAIEmbeddings  # noqa: F401
    from mcp.server.fastmcp import FastMCP  # noqa: F401
    from langchain_mcp_adapters.client import MultiServerMCPClient  # noqa: F401
except Exception as e:
    importlib_ok = False
    print("Error importando:", e)

if importlib_ok:
    print("Dependencias OK. Intérprete:", sys.executable)


Dependencias OK. Intérprete: /home/torres/Escritorio/semestre-9/IA/tarea-19-may/RAG/.venv/bin/python


## 2. Variables desde `.env`

- **`OPENAI_API_KEY`**: cuenta de OpenAI (chat + embeddings).  
- **`LANGCHAIN_API_KEY`**: cuenta de **LangSmith** (trazas); la generas en [smith.langchain.com](https://smith.langchain.com/settings). **No** es la clave de OpenAI.

Copia `.env.example` a `.env` y rellena ambas.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Carga .env desde el directorio de trabajo actual 
_env_path = Path.cwd() / ".env"
loaded = load_dotenv(_env_path, override=False)
if not _env_path.is_file():
    print(f"Aviso: no existe {_env_path.resolve()} — crea el archivo o exporta las variables en el sistema.")
elif not loaded:
    print(f"Aviso: { _env_path } existe pero load_dotenv no aplicó cambios (quizá ya estaban en el entorno).")

# Valores por defecto si no están en .env
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "ia2-rag-chromadb")

_required = ("OPENAI_API_KEY", "LANGCHAIN_API_KEY")
_missing = [k for k in _required if not os.environ.get(k)]
if _missing:
    print("Faltan variables obligatorias:", ", ".join(_missing))
else:
    print("Variables OK: OPENAI_API_KEY y LANGCHAIN_API_KEY están definidas.")


Variables OK: OPENAI_API_KEY y LANGCHAIN_API_KEY están definidas.


## 3. Base de conocimiento (Simón Bolívar)

Hechos cortos **hardcodeados** como lista de strings: el retriever convierte la pregunta en embeddings, busca en Chroma y el agente arma la respuesta con esos fragmentos.


In [3]:
# Los documentos están en server.py (lista DOCUMENTS).
# El servidor MCP los indexa en chroma_db_bolivar/ la primera vez que arranca.
print("Documentos gestionados por server.py")


Documentos gestionados por server.py


## 4. Chroma persistente + embeddings OpenAI

Se usa **`text-embedding-3-small`** (misma `OPENAI_API_KEY` que el chat). Los vectores se guardan en **`chroma_db_bolivar/`** (carpeta dedicada a esta base; si editas los textos, borra la carpeta y vuelve a ejecutar la celda para reindexar).


In [4]:
import sys
from pathlib import Path

SERVER_SCRIPT = Path.cwd() / "server.py"
CHROMA_DIR = Path.cwd() / "chroma_db_bolivar"

if not SERVER_SCRIPT.is_file():
    raise FileNotFoundError(
        f"No se encontró server.py en {SERVER_SCRIPT}. "
        "Crea el archivo antes de continuar."
    )

if CHROMA_DIR.exists() and any(CHROMA_DIR.iterdir()):
    print(f"[Chroma] {CHROMA_DIR.name} existe — el servidor MCP la cargará.")
else:
    print(f"[Chroma] {CHROMA_DIR.name} no existe — el servidor MCP indexará al arrancar.")

SERVER_PATH = str(SERVER_SCRIPT)
print(f"Servidor MCP : {SERVER_PATH}")
print(f"Python       : {sys.executable}")


[Chroma] chroma_db_bolivar no existe — el servidor MCP indexará al arrancar.
Servidor MCP : /home/torres/Escritorio/semestre-9/IA/tarea-19-may/RAG/server.py
Python       : /home/torres/Escritorio/semestre-9/IA/tarea-19-may/RAG/.venv/bin/python


## 5. Agente ReAct con herramienta de recuperación

El modelo **gpt-4o-mini** decide cuándo consultar Chroma (comportamiento tipo ReAct). Con `LANGCHAIN_TRACING_V2=true` las ejecuciones deberían verse en LangSmith. La celda siguiente incluye la pregunta **«¿Quién es Simón Bolívar?»** para mostrar el pipeline completo.


In [6]:
import sys
from pathlib import Path

from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

SERVER_PATH = str(Path.cwd() / "server.py")


async def ejecutar_agente_mcp(pregunta: str) -> str:
    client = MultiServerMCPClient(
        {
            "bolivar_rag": {
                "command": sys.executable,
                "args": [SERVER_PATH],
                "transport": "stdio",
            }
        }
    )
    tools = await client.get_tools()
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    agent = create_react_agent(llm, tools)
    resultado = await agent.ainvoke({"messages": [HumanMessage(content=pregunta)]})
    return resultado["messages"][-1].content


# Pregunta de ejemplo exigida por la tarea: embedding consulta → Chroma (via MCP) → respuesta
pregunta_demostracion = "¿Quién es Simón Bolívar?"
print("Pregunta:", pregunta_demostracion)
respuesta_demo = await ejecutar_agente_mcp(pregunta_demostracion)
print("Respuesta:", respuesta_demo)


Pregunta: ¿Quién es Simón Bolívar?


/tmp/ipykernel_1283344/3229976655.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


Respuesta: Simón Bolívar, conocido como "El Libertador", nació el 24 de julio de 1783 en Caracas, que en ese momento formaba parte del Virreinato de Nueva Granada. Es famoso por su papel militar y político en la independencia de varios países de América Latina, incluyendo Venezuela, Colombia, Ecuador, Perú y Bolivia, que lograron su independencia del imperio español. En 1819, lideró la campaña que culminó en la batalla de Boyacá, un evento decisivo para la independencia de la Nueva Granada.


## 6. Al menos dos preguntas y contextos para RAGAS

Se recuperan contextos con el mismo retriever que usa el agente para alimentar las métricas.


In [7]:
import sys
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PATH = str(Path.cwd() / "server.py")


async def get_contexts_mcp(query: str) -> list:
    """Invoca la tool MCP directamente (sin agente) para obtener contextos para RAGAS."""
    params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(
                "buscar_en_base_vectorial", arguments={"query": query}
            )
            raw = "\n\n".join(
                item.text for item in result.content if hasattr(item, "text")
            )
            # El servidor devuelve "[1] texto\n\n[2] texto\n\n[3] texto"
            contextos = [
                line.split("] ", 1)[1].strip()
                for line in raw.split("\n\n")
                if "] " in line
            ]
            return contextos if contextos else [raw]


preguntas_eval = [
    pregunta_demostracion,
    "¿Qué importancia tuvo la batalla de Boyacá en la carrera de Simón Bolívar?",
]

registros = []
for q in preguntas_eval:
    ctx_docs = await get_contexts_mcp(q)
    respuesta = await ejecutar_agente_mcp(q)
    registros.append({"question": q, "answer": respuesta, "contexts": ctx_docs})
    print("---")
    print("P:", q)
    print("Contextos:", ctx_docs)
    print("R:", respuesta[:500])


/tmp/ipykernel_1283344/3229976655.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


---
P: ¿Quién es Simón Bolívar?
Contextos: ['Simón Bolívar nació el 24 de julio de 1783 en Caracas, entonces parte del Virreinato de Nueva Granada.', 'Es conocido como El Libertador por su papel militar y político en la independencia de Venezuela, Colombia, Ecuador, Perú y Bolivia frente al imperio español.', 'En 1819 lideró la campaña que culminó en la batalla de Boyacá, decisiva para la independencia de la Nueva Granada.']
R: Simón Bolívar, conocido como "El Libertador", nació el 24 de julio de 1783 en Caracas, que en ese momento formaba parte del Virreinato de Nueva Granada. Es famoso por su papel crucial en la independencia de varios países de América del Sur, incluyendo Venezuela, Colombia, Ecuador, Perú y Bolivia, del dominio español. En 1819, lideró la campaña que culminó en la batalla de Boyacá, un evento decisivo para la independencia de la Nueva Granada.


/tmp/ipykernel_1283344/3229976655.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


---
P: ¿Qué importancia tuvo la batalla de Boyacá en la carrera de Simón Bolívar?
Contextos: ['En 1819 lideró la campaña que culminó en la batalla de Boyacá, decisiva para la independencia de la Nueva Granada.', 'Simón Bolívar nació el 24 de julio de 1783 en Caracas, entonces parte del Virreinato de Nueva Granada.', 'Es conocido como El Libertador por su papel militar y político en la independencia de Venezuela, Colombia, Ecuador, Perú y Bolivia frente al imperio español.']
R: La batalla de Boyacá, que tuvo lugar el 7 de agosto de 1819, fue crucial en la carrera de Simón Bolívar, ya que marcó un punto decisivo en la lucha por la independencia de la Nueva Granada (actual Colombia). Esta victoria no solo consolidó su reputación como líder militar, sino que también fue un paso fundamental hacia la liberación de varios países sudamericanos del dominio español. Gracias a esta victoria, Bolívar pudo avanzar en su campaña por la independencia, lo que le valió el título de "E


## 7. RAGAS: faithfulness, answer_relevancy, context_precision

`ground_truth` son referencias cortas manuales para **context_precision**; cámbialas si modificas las preguntas.

En stderr puede aparecer *"LLM returned 1 generations instead of requested 3"*: es aviso interno de RAGAS, no un fallo. La tabla con `faithfulness`, `answer_relevancy` y `context_precision` es el resultado útil.


In [8]:
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper

try:
    from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
except ImportError:
    from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecision

refs = [
    "Simón Bolívar fue un militar y político venezolano nacido en Caracas en 1783, conocido como El Libertador por liderar la independencia de varias naciones sudamericanas del dominio español.",
    "La batalla de Boyacá en 1819 fue decisiva para la independencia de la Nueva Granada y consolidó la posición militar de Bolívar en la campaña libertadora.",
]

eval_ds = Dataset.from_dict(
    {
        "question": [r["question"] for r in registros],
        "answer": [r["answer"] for r in registros],
        "contexts": [r["contexts"] for r in registros],
        "ground_truth": refs[: len(registros)],
    }
)

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

scores = evaluate(
    eval_ds,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
    ],
    llm=ragas_llm,
    embeddings=ragas_emb,
)
print(scores.to_pandas())


/tmp/ipykernel_1283344/854048436.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
/tmp/ipykernel_1283344/854048436.py:8: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
/tmp/ipykernel_1283344/854048436.py:8: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecis

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


                                          user_input  \
0                           ¿Quién es Simón Bolívar?   
1  ¿Qué importancia tuvo la batalla de Boyacá en ...   

                                  retrieved_contexts  \
0  [Simón Bolívar nació el 24 de julio de 1783 en...   
1  [En 1819 lideró la campaña que culminó en la b...   

                                            response  \
0  Simón Bolívar, conocido como "El Libertador", ...   
1  La batalla de Boyacá, que tuvo lugar el 7 de a...   

                                           reference  faithfulness  \
0  Simón Bolívar fue un militar y político venezo...      1.000000   
1  La batalla de Boyacá en 1819 fue decisiva para...      0.888889   

   answer_relevancy  context_precision  
0          0.831003                1.0  
1          0.885611                1.0  
